# Advanced Image Scaling: Bicubic Interpolation

This notebook presents a from-scratch, low-level implementation of bicubic interpolation.

Unlike bilinear interpolation (four nearest neighbors), bicubic uses a 4×4 neighborhood around each sub-pixel location. That yields a smoother approximation of the continuous intensity field, reducing stair-stepping (aliasing) and preserving sharper edges.

### Architecture and Boundary Handling (Edge Cases)

The main engineering challenge with a 4×4 stencil is image borders. This implementation uses **edge-replication padding** (`cv2.copyMakeBorder`) to extend the image virtually and avoid *index out of bounds* errors when estimating partial derivatives ($A_x, A_y, A_{xy}$) at matrix boundaries.

### Computational Complexity (Trade-off)

Bicubic is a classic quality–performance trade-off. Each output pixel requires solving for 16 polynomial coefficients via vector–matrix multiplication with a precomputed inverse matrix ($A^{-1}$), which is substantially heavier than linear methods.

In [ ]:
import cv2
import os
from matplotlib import pyplot as plt
import numpy as np

# Download source files and precomputed inverse matrix
if not os.path.exists("parrot.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/parrot.bmp --no-check-certificate
if not os.path.exists("clock.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/clock.bmp --no-check-certificate
if not os.path.exists("chessboard.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/chessboard.bmp --no-check-certificate

if not os.path.exists("ainvert.py") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/ainvert.py --no-check-certificate

import ainvert

def bicubic_interpolation(img, scale_factor):
    img_float = img.astype(np.float32)
    
    h, w = img_float.shape[:2]
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
    new_img = np.zeros((new_h, new_w), dtype=np.float32)
    
    # Boundary padding for edge pixels
    pad_img = cv2.copyMakeBorder(img_float, 2, 2, 2, 2, cv2.BORDER_REPLICATE)
    
    A_inv = ainvert.A_invert
    
    for i in range(new_h):
        for j in range(new_w):
            x = i / scale_factor
            y = j / scale_factor
            
            dx = x - np.floor(x)
            dy = y - np.floor(y)
            
            # Index offset accounting for added padding
            i0 = int(np.floor(x)) + 2 - 1
            j0 = int(np.floor(y)) + 2 - 1

            neighborhood = pad_img[i0:i0+4, j0:j0+4]

            # Sample grid nodes
            A = neighborhood[1, 1]
            B = neighborhood[2, 1]
            C = neighborhood[2, 2]
            D = neighborhood[1, 2]

            # Estimate partial derivatives
            Ax = (neighborhood[2, 1] - neighborhood[0, 1]) / 2.0
            Ay = (neighborhood[1, 2] - neighborhood[1, 0]) / 2.0
            Axy = (neighborhood[2, 2] - neighborhood[0, 1] - neighborhood[1, 0] + neighborhood[1, 1]) / 4.0
            
            Bx = (neighborhood[3, 1] - neighborhood[1, 1]) / 2.0
            By = (neighborhood[2, 2] - neighborhood[2, 0]) / 2.0
            Bxy = (neighborhood[3, 2] - neighborhood[1, 1] - neighborhood[2, 0] + neighborhood[2, 1]) / 4.0
            
            Cx = (neighborhood[3, 2] - neighborhood[1, 2]) / 2.0
            Cy = (neighborhood[2, 3] - neighborhood[2, 1]) / 2.0
            Cxy = (neighborhood[3, 3] - neighborhood[1, 2] - neighborhood[2, 1] + neighborhood[2, 2]) / 4.0
            
            Dx = (neighborhood[2, 2] - neighborhood[0, 2]) / 2.0
            Dy = (neighborhood[1, 3] - neighborhood[1, 1]) / 2.0
            Dxy = (neighborhood[2, 3] - neighborhood[0, 2] - neighborhood[1, 1] + neighborhood[1, 2]) / 4.0
            
            x_vec = np.array([A, B, D, C, Ax, Bx, Dx, Cx, Ay, By, Dy, Cy, Axy, Bxy, Dxy, Cxy])
            
            # Solve the linear system
            a = A_inv @ x_vec
            
            value = 0.0
            idx = 0
            for q in range(4):     
                for p in range(4): 
                    value += a[idx] * (dx ** p) * (dy ** q)
                    idx += 1
            
            new_img[i, j] = value
            
    return np.clip(new_img, 0, 255).astype(np.uint8)

img = cv2.imread('parrot.bmp', cv2.IMREAD_GRAYSCALE)

scale = 1.5
result = bicubic_interpolation(img, scale)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(result, cmap='gray')
plt.title(f'Bicubic Interpolation (Scale: {scale}x)')
plt.axis('off')

plt.tight_layout()
plt.show()

### Cleaning the Workspace

In [ ]:
import shutil
import glob

trash_files = glob.glob('*.bmp*') + glob.glob('ainvert.py*')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

if os.path.exists('__pycache__'):
    shutil.rmtree('__pycache__')

print("Environment cleared of temporary files.")